# Task 1.1 — Core Contribution / Architecture

**Paper:** "Object Detection with Discriminatively Trained Part-Based Models"  
**Authors:** Felzenszwalb, Girshick, McAllester, Ramanan (2010)  
**Venue:** IEEE Transactions on Pattern Analysis and Machine Intelligence (TPAMI)

---

## Overview

This notebook provides a step-by-step walkthrough of the Deformable Part Model (DPM) pipeline — from input image to final object detection output. The DPM represents objects as a coarse root template combined with higher-resolution part templates that can deform relative to their expected positions. The entire system is trained discriminatively using a Latent SVM formulation.

---

## Step 1: Feature Pyramid Construction

### Explanation

The first stage of the DPM pipeline is to compute a **feature pyramid** over the input image. The image is rescaled to multiple resolutions (typically around 30 levels), and at each resolution, a dense grid of **Histogram of Oriented Gradients (HOG)** features is extracted.

HOG descriptors capture local gradient orientation statistics in fixed-size cells (typically 8×8 pixels). The feature map at each scale of the pyramid has shape (H × W × 31), where 31 is the dimensionality of the HOG descriptor per cell used in this paper (an extended variant of the original Dalal & Triggs HOG).

The pyramid is constructed at **two resolutions per octave** — the root filter operates at one level, and the part filters operate at **twice the spatial resolution** (one octave higher). This means each part filter cell corresponds to a 4×4 pixel region while the root filter cell corresponds to 8×8 pixels.

### Paper Reference
- **Section 2.1** — describes the HOG feature representation and the multi-scale feature pyramid.
- **Figure 3** — illustrates the feature pyramid with root and part filter levels.
- **Section 2.4** — details the extended 31-dimensional HOG variant.

### Purpose
The feature pyramid enables the model to detect objects at **multiple scales** without needing to resize the filter templates. By scanning filters at each pyramid level, the detector naturally handles objects of different sizes in the image.

---

## Step 2: Root Filter Detection

### Explanation

The **root filter** is a single HOG template trained to capture the overall shape of the object category (e.g., a person, car, or bicycle). It is a linear filter — essentially a weight vector $F_0$ of size $(w_0 \times h_0 \times 31)$ — that is convolved with the feature maps at each level of the pyramid.

At each position $(x, y)$ and scale $l$, the root filter response is computed as a dot product:

$$\text{score}_{\text{root}}(x, y, l) = F_0 \cdot \phi(H, (x, y, l))$$

where $\phi(H, (x, y, l))$ denotes the HOG feature vector extracted from pyramid level $l$ at position $(x, y)$.

This produces a **score map** at each scale, indicating how well the root template matches each spatial location.

### Paper Reference
- **Section 2.1** — defines the root filter and its scoring function.
- **Equation 1** — filter response as dot product of filter weights and feature subwindow.
- **Section 3** — star model architecture with root filter.

### Purpose
The root filter provides a **coarse, holistic detection** of the object. It captures the global shape but cannot model fine-grained details or articulations — this is where part filters come in.

---

## Step 3: Part Filter Detection

### Explanation

In addition to the root filter, the model includes **$n$ part filters** $F_1, \ldots, F_n$ (typically 6–8 parts). Each part filter operates at **twice the spatial resolution** of the root filter (one level higher in the feature pyramid). This means parts are modeled with finer detail than the root.

Each part filter is smaller than the root filter and is designed to capture a local component of the object (e.g., the head, torso, or limbs of a person). Like the root filter, each part filter is convolved with the higher-resolution feature map to produce a part-specific response map:

$$\text{score}_{\text{part}_i}(x, y, l) = F_i \cdot \phi(H, (x, y, l))$$

Each part has an **anchor position** $(v_{ix}, v_{iy})$ defined relative to the root filter. This anchor specifies where the part is *expected* to be located.

### Paper Reference
- **Section 3** — defines star-structured part-based models.
- **Figure 4** — visualizes root and part filters for different object categories.
- **Equation 2** — scoring of part filters.

### Purpose
Part filters capture **local, high-resolution appearance** of object components. By modeling parts separately, the system can handle objects whose internal structure varies (e.g., different body poses).

---

## Step 4: Deformation Modeling

### Explanation

The key innovation of the DPM is allowing parts to **deform** — i.e., shift away from their expected anchor positions — at a cost determined by a learned deformation penalty. For each part $i$, a deformation cost is computed using a **quadratic function** of the displacement $(dx, dy)$ from the anchor:

$$\text{deformation\_cost}_i(dx, dy) = d_i \cdot (dx, dy, dx^2, dy^2)$$

where $d_i = (d_{i1}, d_{i2}, d_{i3}, d_{i4})$ is a learned 4-dimensional deformation coefficient vector for part $i$. The components $d_{i3}$ and $d_{i4}$ (coefficients of the squared terms) are constrained to be non-positive, so the deformation cost is a **concave quadratic** — it penalizes large displacements.

The overall score for placing part $i$ at position $(x_i, y_i)$ when the root is at $(x_0, y_0)$ is:

$$\text{score}_i(x_i, y_i) = F_i \cdot \phi(H, (x_i, y_i)) - d_i \cdot \Phi_d(dx_i, dy_i)$$

where $\Phi_d(dx, dy) = (dx, dy, dx^2, dy^2)$ is the deformation feature vector.

The **optimal placement** of each part is found via the **generalized distance transform**, which efficiently computes the maximum part score over all possible displacements in $O(n)$ time per row/column of the score map.

### Paper Reference
- **Section 3, Equation 3** — defines the deformation cost.
- **Section 2.3** — describes the distance transform trick for efficient part placement.
- **Figure 5** — visualizes learned deformation patterns.

### Purpose
Deformation modeling allows the DPM to handle **articulated and deformable objects**. A person's arms can be in different positions; a car may appear slightly different from various viewpoints. The deformation penalty balances appearance matching with spatial consistency.

---

## Step 5: Latent SVM Training

### Explanation

The complete model for a single component (one root + its parts) produces a score:

$$\text{score}(x_0, \ldots, x_n, l) = F_0 \cdot \phi(H, x_0, l) + \sum_{i=1}^{n}\left[F_i \cdot \phi(H, x_i, l) - d_i \cdot \Phi_d(dx_i, dy_i)\right] + b$$

where $b$ is a bias term. The model parameters $\beta = (F_0, F_1, \ldots, F_n, d_1, \ldots, d_n, b)$ are learned discriminatively.

The challenge is that during training, the **part positions are not annotated** — only bounding boxes are available. Part placements are therefore **latent variables**. The training problem becomes a **Latent SVM** (also called MI-SVM or semi-convex optimization):

$$\min_{\beta} \frac{1}{2}\|\beta\|^2 + C \sum_i \max(0, 1 - y_i \cdot f_\beta(x_i))$$

For positive examples, the latent variables (part positions) are chosen to **maximize** the score. For negative examples, the score is computed by maximizing over all positions.

The optimization alternates between:
1. **Fixing latent variables** for positive examples (choosing best part placements) → standard SVM.
2. **Updating model parameters** using the fixed assignments → convex optimization.

This coordinate-descent approach converges to a local optimum.

### Paper Reference
- **Section 4** — defines Latent SVM formulation.
- **Equation 6** — the semi-convex objective function.
- **Section 4.1** — describes the coordinate descent optimization.

### Purpose
Latent SVM allows the model to be trained **without explicit part annotations**, learning part positions jointly with filter weights. This is critical because annotating individual parts is expensive and subjective.

---

## Step 6: Hard Negative Mining

### Explanation

Training the SVM on the full set of negative windows is computationally infeasible — there are millions of possible windows in a typical image. The paper uses **hard negative mining** (also called bootstrapping):

1. Train an initial model on all positive examples and a random subset of negatives.
2. Run the detector on all training images.
3. Collect **false positives** (negative windows that score above the detection threshold) — these are the *hard negatives*.
4. Add hard negatives to the training set and retrain.
5. Repeat until convergence (no new hard negatives found).

The key theoretical insight (Section 4.2) is that maintaining a **cache of hard negatives** and iteratively retraining is equivalent to optimizing the full objective, because any negative not in the cache has zero loss (scores below the margin).

### Paper Reference
- **Section 4.2** — describes the data-mining (hard negative mining) procedure.
- **Algorithm description** at end of Section 4 — outlines the full training loop.

### Purpose
Hard negative mining focuses the learning on the **most confusing** background regions, dramatically improving discriminative power without needing to process all possible negatives simultaneously.

---

## Step 7: Final Object Detection

### Explanation

At test time, the full detection pipeline proceeds as:

1. Compute the HOG feature pyramid for the test image.
2. Score the **root filter** at every position and scale.
3. At twice the resolution, score each **part filter** and compute optimal placements using the distance transform (accounting for deformation costs).
4. Combine root + part scores + deformation costs + bias to get the overall score at each (position, scale).
5. The model may have **multiple components** (mixture model) to handle different viewpoints (e.g., frontal vs. side view of a car). Each component is scored independently.
6. Apply a **score threshold** to obtain candidate detections.
7. Apply **non-maximum suppression (NMS)** — using a greedy procedure based on overlap — to remove duplicate detections.
8. Output the final bounding boxes with confidence scores.

### Paper Reference
- **Section 3.2** — mixture models with multiple components.
- **Section 5** — describes the full detection system.
- **Section 6** — experimental results on PASCAL VOC.

### Purpose
The final detection step aggregates all model components and produces a ranked list of detections. Non-maximum suppression ensures each object instance generates only one detection box.

---

## Summary

**What problem does the paper solve?**  
The paper addresses the problem of **object detection** in natural images — specifically, localizing and recognizing objects from categories like person, car, bicycle, etc., under significant variation in viewpoint, pose, and appearance.

**Why does the proposed method improve over previous approaches?**  
Prior methods (e.g., Dalal & Triggs HOG detector, Viola-Jones) used either **rigid templates** or **simple cascades** that could not model articulation or deformation. The DPM improves upon these by:

1. **Deformable parts**: Instead of a single rigid template, the model uses a root filter for global shape plus multiple part filters that can move, capturing local variations such as body pose or viewpoint changes.
2. **Discriminative training**: Unlike generative approaches, the Latent SVM training directly optimizes detection performance using hinge loss.
3. **Latent variable formulation**: Part locations are treated as latent variables, eliminating the need for expensive part-level annotations.
4. **Multi-component mixture**: Different components handle different aspects/viewpoints of an object category.

The DPM achieved state-of-the-art results on the **PASCAL VOC 2006, 2007, and 2008** benchmarks, winning the VOC challenge in multiple years and establishing a new paradigm for pre-deep-learning object detection.